# 03 - LangChain Proxy Deep Dive

Your Open WebUI traffic is configured to flow through `langchain-demo` at `/ollama/api/*`.

Recommended pre-step (separate terminal):
```bash
kubectl port-forward -n llm-observability svc/langchain-demo 8000:8000
kubectl port-forward -n llm-observability svc/ollama 11434:11434
```

In [ ]:
import os
import sys
from pathlib import Path

expected = "/usr/local/bin/python3.11"
print(f"Python executable: {sys.executable}")
print(f"Python version   : {sys.version.split()[0]}")
if Path(sys.executable).resolve().as_posix() != expected:
    print(f"WARNING: This notebook is designed for {expected}.")
    print("Switch the notebook kernel to Python 3.11 before continuing.")
else:
    print("Kernel check passed.")


In [ ]:
import importlib
import subprocess
import sys

packages = ["requests", "pandas", "matplotlib", "pyyaml", "langsmith", "tabulate"]
missing = []
for pkg in packages:
    module_name = "yaml" if pkg == "pyyaml" else pkg
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pkg)

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("All common packages are already installed.")


In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd
import requests

PROXY_BASE = "http://localhost:8000"
OLLAMA_DIRECT = "http://localhost:11434"
PROXY_PORT_FORWARD_CMD = "kubectl port-forward -n llm-observability svc/langchain-demo 8000:8000"
OLLAMA_PORT_FORWARD_CMD = "kubectl port-forward -n llm-observability svc/ollama 11434:11434"
MODEL_NAME = "gemma3-1b-it-gguf-local"


def url_available(url: str, timeout: int = 3) -> bool:
    try:
        r = requests.get(url, timeout=timeout)
        return r.status_code < 500
    except Exception:
        return False


PROXY_AVAILABLE = url_available(f"{PROXY_BASE}/healthz")
DIRECT_AVAILABLE = url_available(f"{OLLAMA_DIRECT}/api/tags")
print("PROXY_AVAILABLE :", PROXY_AVAILABLE)
print("DIRECT_AVAILABLE:", DIRECT_AVAILABLE)
if not PROXY_AVAILABLE:
    print("Proxy preflight failed. In another terminal run:")
    print(f"  {PROXY_PORT_FORWARD_CMD}")
if not DIRECT_AVAILABLE:
    print("Direct Ollama preflight failed. In another terminal run:")
    print(f"  {OLLAMA_PORT_FORWARD_CMD}")


## Proxy Configuration and Health


In [ ]:
if not PROXY_AVAILABLE:
    print("Skipping proxy configuration check: langchain-demo is not reachable on localhost:8000")
else:
    for endpoint in ["/", "/healthz", "/config"]:
        try:
            r = requests.get(f"{PROXY_BASE}{endpoint}", timeout=15)
            print(endpoint, "->", r.status_code)
            if "application/json" in r.headers.get("content-type", ""):
                print(r.json())
            else:
                print(r.text[:500])
        except Exception as exc:
            print(endpoint, "->", exc)


## /invoke Endpoint


In [ ]:
invoke_payload = {
    "prompt": "Return 3 bullet points about tracing proxy architecture.",
    "system": "You are concise.",
}

if not PROXY_AVAILABLE:
    print("Skipping /invoke test: langchain-demo is not reachable on localhost:8000")
else:
    try:
        r = requests.post(f"{PROXY_BASE}/invoke", json=invoke_payload, timeout=180)
        r.raise_for_status()
        r.json()
    except Exception as exc:
        print(f"/invoke request failed: {exc}")


## Proxy-Compatible Ollama Endpoints


In [ ]:
if not PROXY_AVAILABLE:
    print("Skipping proxy Ollama endpoint tests: langchain-demo is not reachable on localhost:8000")
else:
    try:
        tags = requests.get(f"{PROXY_BASE}/ollama/api/tags", timeout=20)
        tags.raise_for_status()
        print("models via proxy:")
        for item in tags.json().get("models", []):
            print("-", item.get("name"))

        chat_body = {
            "model": MODEL_NAME,
            "stream": False,
            "messages": [{"role": "user", "content": "Summarize request forwarding in 2 lines."}],
        }
        chat = requests.post(f"{PROXY_BASE}/ollama/api/chat", json=chat_body, timeout=180)
        chat.raise_for_status()
        print(chat.json().get("message", {}).get("content", "")[:1200])
    except Exception as exc:
        print(f"Proxy Ollama endpoint test failed: {exc}")



## Direct vs Proxy Latency Comparison


In [ ]:
# Keep this benchmark intentionally light so langchain-demo health probes stay responsive.
MAX_ITERS = 4
REQUEST_TIMEOUT_SECONDS = 120
SLEEP_BETWEEN_CALLS_SECONDS = 2
MAX_PREDICT = 64


def timed_chat(url, path):
    payload = {
        "model": MODEL_NAME,
        "stream": False,
        "messages": [{"role": "user", "content": "In one sentence, explain why request IDs help debugging."}],
        # Cap generated tokens to avoid very long calls that can trigger pod probe failures.
        "options": {"num_predict": MAX_PREDICT},
    }
    start = time.perf_counter()
    try:
        r = requests.post(url, json=payload, timeout=REQUEST_TIMEOUT_SECONDS)
        elapsed = (time.perf_counter() - start) * 1000
        return r.status_code, elapsed, None
    except Exception as exc:
        elapsed = (time.perf_counter() - start) * 1000
        return -1, elapsed, str(exc)


rows = []
if not (DIRECT_AVAILABLE or PROXY_AVAILABLE):
    print("Skipping latency comparison: neither direct Ollama nor proxy endpoint is reachable")
else:
    for i in range(MAX_ITERS):
        if DIRECT_AVAILABLE:
            s1, t1, e1 = timed_chat(f"{OLLAMA_DIRECT}/api/chat", "direct")
            rows.append({"iteration": i + 1, "path": "direct", "status": s1, "latency_ms": round(t1, 2), "error": e1})
            time.sleep(SLEEP_BETWEEN_CALLS_SECONDS)

        if PROXY_AVAILABLE:
            if not url_available(f"{PROXY_BASE}/healthz", timeout=2):
                rows.append(
                    {
                        "iteration": i + 1,
                        "path": "proxy",
                        "status": -1,
                        "latency_ms": 0.0,
                        "error": "proxy /healthz is not reachable before request; stopping proxy benchmark",
                    }
                )
                PROXY_AVAILABLE = False
            else:
                s2, t2, e2 = timed_chat(f"{PROXY_BASE}/ollama/api/chat", "proxy")
                rows.append({"iteration": i + 1, "path": "proxy", "status": s2, "latency_ms": round(t2, 2), "error": e2})
                if e2 and ("Connection aborted" in e2 or "Connection refused" in e2):
                    print("Proxy connection dropped; stopping further proxy iterations.")
                    PROXY_AVAILABLE = False
                time.sleep(SLEEP_BETWEEN_CALLS_SECONDS)

if rows:
    cmp_df = pd.DataFrame(rows)
else:
    cmp_df = pd.DataFrame(columns=["iteration", "path", "status", "latency_ms", "error"])

cmp_df



In [ ]:
if cmp_df.empty:
    print("No comparison rows available.")
else:
    summary = cmp_df.groupby("path")["latency_ms"].agg(["mean", "median", "max"]).round(2)
    summary


In [ ]:
if cmp_df.empty:
    print("No comparison rows available for plotting.")
else:
    plot_df = cmp_df.pivot(index="iteration", columns="path", values="latency_ms")
    ax = plot_df.plot(figsize=(10, 4), marker="o", title="Direct vs Proxy Latency")
    ax.set_ylabel("Latency (ms)")
    plt.show()


## Quick Pod Log Snapshot


In [ ]:
import subprocess

cmd = "kubectl logs -n llm-observability deploy/langchain-demo --tail=80"
proc = subprocess.run(cmd, shell=True, text=True, capture_output=True)
print(proc.stdout if proc.stdout else proc.stderr)


## Done
Continue with `04-langsmith-tracing-setup.ipynb` to validate traces inside your LangSmith project.
